# Week C — resolve identities

**Goal:** test invariant I2 against real data. The schema asserts that every `ModificationSite` is keyed on a UniProt accession *and a sequence version*, because residue numbering is meaningless without one.

This notebook checks whether that is a real problem or a theoretical one. For twenty sites, it asks UniProt for the sequence and verifies that the position MaxQuant reported actually is a lysine.

**Why this one matters differently.** Weeks A and B were analysis — throwaway once the platform exists. This is the resolver, and it goes into the product nearly unchanged.

Expect failures. Obsolete accessions, merged entries, and sequences amended since the 2018-era FASTA the search used are all likely, and each is exactly what I2 exists to catch.

## Step 1 — load the site table

In [ ]:
import pandas as pd, numpy as np, requests, os, time, json

URL = ("https://ftp.pride.ebi.ac.uk/pride/data/archive/2022/02/"
       "PXD018299/HAP1_USP18KO_GlyGlyKSites.txt")
LOCAL = "HAP1_USP18KO_GlyGlyKSites.txt"

if not os.path.exists(LOCAL):
    with requests.get(URL, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(LOCAL, "wb") as fh:
            for chunk in r.iter_content(chunk_size=1 << 20):
                fh.write(chunk)

df = pd.read_csv(LOCAL, sep="\t", low_memory=False)
clean = df[(df["Reverse"] != "+") & (df["Potential contaminant"] != "+")].copy()
print(f"{len(clean):,} sites after filtering")

## Step 2 — what the file claims about each site

MaxQuant gives a razor pick in `Protein` and the full candidate set in `Proteins`. `Position` is the residue number within the protein, `Amino acid` should be K.

Note there is **no sequence version anywhere in this file**. That is the gap I2 identifies: the position is stated against a sequence the file does not identify.

In [ ]:
cols = [c for c in ["Protein", "Proteins", "Gene names", "Position",
                    "Amino acid", "Sequence window", "Localization prob"]
        if c in clean.columns]
print("available:", cols, "\n")
clean[cols].head(10)

## Step 3 — take a sample

Twenty sites, biased toward well-localised ones so any failure is a resolution problem rather than a localisation problem.

In [ ]:
N = 20

sample = (clean[clean["Localization prob"] >= 0.9]
          .sample(n=N, random_state=0)[cols]
          .reset_index(drop=True))
sample

## Step 4 — ask UniProt

The REST API returns JSON per accession. Three fields matter:

- `sequence.value` — the amino acid string
- `entryAudit.sequenceVersion` — **the number the whole schema hangs on**
- `entryType` — whether the entry is reviewed (Swiss-Prot) or unreviewed (TrEMBL)

A polite delay between requests; UniProt is a public service.

In [ ]:
def fetch_uniprot(accession):
    acc = accession.split("-")[0]          # strip isoform suffix, e.g. Q00341-2
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.json"
    try:
        r = requests.get(url, timeout=30)
    except Exception as e:
        return {"status": "network_error", "detail": str(e)}

    if r.status_code == 404:
        return {"status": "not_found"}
    if r.status_code != 200:
        return {"status": f"http_{r.status_code}"}

    d = r.json()
    return {
        "status": "ok",
        "primary_accession": d.get("primaryAccession"),
        "entry_type": d.get("entryType", ""),
        "sequence": d.get("sequence", {}).get("value", ""),
        "sequence_version": d.get("entryAudit", {}).get("sequenceVersion"),
        "last_seq_update": d.get("entryAudit", {}).get("lastSequenceUpdateDate"),
        "gene": (d.get("genes") or [{}])[0].get("geneName", {}).get("value"),
    }

test = fetch_uniprot("P05161")   # ISG15
print(json.dumps({k: (v[:60] + "..." if k == "sequence" else v)
                  for k, v in test.items()}, indent=2))

## Step 5 — validate each site

The check: is the residue at `Position` in the current UniProt sequence actually a lysine?

Four possible outcomes, and each maps to something in the schema:

| Outcome | Meaning | Schema |
|---|---|---|
| `ok` | Position holds a K | `ModificationSite` created |
| `wrong_residue` | Position holds something else | Sequence changed since the search — I2 firing |
| `out_of_range` | Position beyond sequence end | Sequence shortened, or wrong accession |
| `not_found` | Accession retired or merged | Needs the ID-mapping service |

In [ ]:
results = []

for _, row in sample.iterrows():
    acc = str(row["Protein"])
    pos = int(row["Position"]) if not pd.isna(row["Position"]) else None
    info = fetch_uniprot(acc)
    time.sleep(0.4)

    rec = {"accession": acc, "position": pos,
           "gene_in_file": row.get("Gene names"),
           "status": info["status"]}

    if info["status"] == "ok":
        seq = info["sequence"]
        rec.update({
            "seq_version": info["sequence_version"],
            "last_seq_update": info["last_seq_update"],
            "reviewed": "UniProtKB reviewed" in info["entry_type"],
            "seq_length": len(seq),
            "gene_in_uniprot": info["gene"],
        })
        if pos is None or pos > len(seq):
            rec["check"] = "out_of_range"
            rec["residue"] = None
        else:
            aa = seq[pos - 1]
            rec["residue"] = aa
            rec["check"] = "ok" if aa == "K" else "wrong_residue"
    else:
        rec["check"] = info["status"]

    results.append(rec)
    print(f"{acc:12s} pos {str(pos):>5s}  {rec['check']}")

res = pd.DataFrame(results)
print("\n--- summary ---")
print(res["check"].value_counts())

## Step 6 — the numbers that matter

In [ ]:
n = len(res)
ok = (res["check"] == "ok").sum()
print(f"validated:        {ok}/{n} ({100*ok/n:.0f}%)")

if "reviewed" in res:
    rev = res["reviewed"].sum()
    print(f"reviewed entries: {rev}/{n} — the rest are TrEMBL razor picks")

if "seq_version" in res:
    print(f"\nsequence versions in play: "
          f"{sorted(res['seq_version'].dropna().unique().tolist())}")
    print("\nlatest sequence updates:")
    print(res["last_seq_update"].dropna().sort_values().tail(5).to_string())

print("\nfailures:")
bad = res[res["check"] != "ok"]
print(bad.to_string() if len(bad) else "none")

## Step 7 — the version question

The site table has no sequence version, so the positions were computed against whatever FASTA the search used — a UniProt release from around 2018–2019.

**Check whether any of these sequences have been amended since.** A `lastSequenceUpdateDate` after the search means the position may have shifted, and a site that still validates as K might now be a *different* lysine.

That silent failure is precisely why I2 puts the version in the primary key.

In [ ]:
SEARCH_DATE = "2019-01-01"   # approximate; the deposit predates 2020 publication

if "last_seq_update" in res:
    upd = res.dropna(subset=["last_seq_update"]).copy()
    upd["amended_since_search"] = upd["last_seq_update"] > SEARCH_DATE
    n_amended = upd["amended_since_search"].sum()
    print(f"sequences amended since ~{SEARCH_DATE}: {n_amended} of {len(upd)}")
    if n_amended:
        print("\nthese positions may refer to different residues than intended:")
        print(upd[upd["amended_since_search"]]
              [["accession", "position", "seq_version", "last_seq_update", "check"]]
              .to_string())
    else:
        print("none in this sample — but the sample is 20 of 2,298.")

## Step 8 — reviewed versus unreviewed

Week A found the razor pick landing on `A0A024R4E5` (TrEMBL) rather than `Q00341` (reviewed Swiss-Prot) for vigilin. This checks how often the candidate set contains a reviewed entry that the razor pick passed over.

The answer decides whether resolution should prefer reviewed entries — recorded as `ProteinAssignment` basis, never applied silently.

In [ ]:
def is_reviewed(acc):
    info = fetch_uniprot(acc)
    time.sleep(0.4)
    return info.get("status") == "ok" and "reviewed" in info.get("entry_type", "")

check = sample.head(8)
for _, row in check.iterrows():
    picked = str(row["Protein"])
    candidates = [c.strip() for c in str(row["Proteins"]).split(";") if c.strip()]
    reviewed = [c for c in candidates[:6] if is_reviewed(c)]
    flag = "" if picked in reviewed else "  <- picked an unreviewed entry"
    print(f"picked {picked:14s} of {len(candidates):2d} candidates; "
          f"reviewed among first 6: {reviewed}{flag}")

## Step 9 — record what you found

Into **Measured findings** in `ROADMAP.md`:

1. What fraction of positions validated as lysine against current UniProt.
2. Which failed, and why — retired accession, shortened sequence, wrong residue.
3. How many sequences have been amended since the original search.
4. How often the razor pick is an unreviewed entry when a reviewed one was available.

**The decision this informs.** If validation is near 100%, I2 is cheap insurance and resolution can proceed optimistically. If a meaningful fraction fails, the resolver needs UniProt's ID-mapping service and possibly historical sequence retrieval — which is currently open question 1 in `ARCHITECTURE.md`, and this answers it with data.

---

**This code is not throwaway.** Steps 4 and 5 are the resolver module in `bzk/resolve/`, minus the caching layer. When you have a real environment, this is the first thing to port.